# 10 - 导出与部署**学习目标：**- 了解不同模型导出格式的特点- 学会将模型导出为 ONNX 和 CoreML- 理解推理优化的基本概念- 在 Mac 上使用 CoreML 加速推理---

## 1. 为什么需要导出训练好的 `.pt` 文件是 PyTorch 格式，但部署时需要其他格式：| 格式 | 平台 | 特点 ||------|------|------|| **PyTorch (.pt)** | 通用 | 训练用，依赖 Python 环境 || **ONNX** | 通用 | 跨框架标准，兼容性最好 || **CoreML (.mlpackage)** | Apple | Mac/iOS 原生加速 || **TensorRT** | NVIDIA | GPU 推理最快 || **TFLite** | 移动端 | Android/嵌入式 || **NCNN** | 移动端 | 轻量级，适合移动端 |**Mac 上推荐**：CoreML（M 芯片原生加速）

## 2. 导出为 ONNX**ONNX = Open Neural Network Exchange**- 微软和 Facebook 联合推出的标准格式- 几乎所有推理框架都支持

In [ ]:
from ultralytics import YOLOfrom pathlib import Path# 加载训练好的模型（或预训练模型）model = YOLO("yolo11n.pt")# 导出为 ONNXonnx_path = model.export(    format="onnx",    imgsz=640,    simplify=True,     # 简化计算图    opset=17,          # ONNX opset 版本)print(f"ONNX 模型已导出: {onnx_path}")print(f"文件大小: {Path(onnx_path).stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
# 使用 ONNX 模型推理import onnxruntime as ortimport numpy as np# 加载 ONNX 模型session = ort.InferenceSession(str(onnx_path))# 查看输入输出信息print("模型输入：")for inp in session.get_inputs():    print(f"  {inp.name}: {inp.shape} ({inp.type})")print("\n模型输出：")for out in session.get_outputs():    print(f"  {out.name}: {out.shape}")

## 3. 导出为 CoreML（Mac 推荐）**CoreML** 是 Apple 的机器学习框架：- M 芯片 Neural Engine 原生加速- 无需额外依赖- iOS/macOS 原生支持

In [ ]:
from ultralytics import YOLOmodel = YOLO("yolo11n.pt")# 导出为 CoreMLcoreml_path = model.export(    format="coreml",    imgsz=640,    nms=True,          # 包含 NMS 后处理)print(f"CoreML 模型已导出: {coreml_path}")

## 4. 推理速度对比

In [ ]:
import timefrom ultralytics import YOLOfrom PIL import Imagefrom pathlib import Path# 准备测试图片img_dir = Path("../data/coco128/images/train2017")test_img = str(sorted(img_dir.glob("*.jpg"))[0])img = Image.open(test_img)# PyTorch 推理model_pt = YOLO("yolo11n.pt")model_pt(test_img, verbose=False)  # 预热start = time.time()for _ in range(10):    r = model_pt(test_img, verbose=False)pt_time = (time.time() - start) / 10print(f"PyTorch 推理: {pt_time*1000:.1f} ms/张")# ONNX 推理try:    import onnxruntime as ort    session = ort.InferenceSession(str(onnx_path))        # 预处理    img_resized = img.resize((640, 640))    img_np = np.array(img_resized).astype(np.float32) / 255.0    img_np = img_np.transpose(2, 0, 1)[np.newaxis]        start = time.time()    for _ in range(10):        _ = session.run(None, {session.get_inputs()[0].name: img_np})    onnx_time = (time.time() - start) / 10        print(f"ONNX 推理:    {onnx_time*1000:.1f} ms/张")except Exception as e:    print(f"ONNX 推理失败: {e}")

## 5. 模型优化技巧### 量化 (Quantization)将浮点权重 (FP32) 转换为低精度 (FP16/INT8)：```FP32: 4 bytes/param → FP16: 2 bytes → INT8: 1 byte模型大小: 6MB → 3MB → 1.5MB速度: 慢 → 快 → 更快精度: 高 → 略降 → 可能下降```

In [ ]:
# FP16 量化导出model = YOLO("yolo11n.pt")# 导出 FP16 ONNXonnx_fp16_path = model.export(    format="onnx",    imgsz=640,    half=True,  # FP16 半精度)print(f"FP16 ONNX 已导出: {onnx_fp16_path}")

### 输入尺寸优化更小的输入 = 更快的推理，但精度可能下降：| 输入尺寸 | 速度 | 精度 | 适用场景 ||----------|------|------|----------|| 320 | 最快 | 较低 | 实时视频、移动端 || 640 | 标准 | 标准 | 通用场景 || 1280 | 较慢 | 更高 | 小目标检测 |

## 6. 实际部署示例### Python 服务端部署

In [ ]:
# 最简部署方式 - 直接用 ultralyticsfrom ultralytics import YOLOfrom PIL import Imagefrom pathlib import Pathclass YOLODetector:    """简单的 YOLO 检测器封装"""        def __init__(self, model_path="yolo11n.pt", conf=0.25, iou=0.5):        self.model = YOLO(model_path)        self.conf = conf        self.iou = iou        def detect(self, image):        """检测图片中的目标"""        results = self.model(image, conf=self.conf, iou=self.iou, verbose=False)        result = results[0]                detections = []        for box in result.boxes:            detections.append({                'class': self.model.names[int(box.cls[0])],                'confidence': float(box.conf[0]),                'bbox': box.xyxy[0].tolist(),  # [x1, y1, x2, y2]            })        return detections        def detect_and_save(self, image_path, output_path):        """检测并保存标注后的图片"""        results = self.model(image_path, conf=self.conf, verbose=False)        annotated = results[0].plot()        Image.fromarray(annotated[..., ::-1]).save(output_path)        return output_path# 使用示例detector = YOLODetector(conf=0.3)# 检测img_dir = Path("../data/coco128/images/train2017")test_img = str(sorted(img_dir.glob("*.jpg"))[0])detections = detector.detect(test_img)print(f"检测到 {len(detections)} 个目标：")for d in detections:    print(f"  {d['class']}: {d['confidence']:.2%}")

## 7. 部署清单训练好模型后，部署步骤：```✅ 1. 训练并选择最佳模型（best.pt）✅ 2. 在测试集上评估，确认精度达标✅ 3. 导出为目标格式（ONNX/CoreML）✅ 4. 测试导出模型的推理速度和精度✅ 5. 集成到应用中✅ 6. 监控线上表现，持续优化```---**上一节**: [09 - 自定义数据集](09_custom_dataset.ipynb)**🎉 恭喜完成 YOLO 学习全部课程！**